In [1]:
!pip install flash-attn --no-build-isolation

     ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/8.4 MB 9.3 MB/s eta 0:00:01

     ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/8.4 MB 40.4 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 70.5 MB/s eta 0:00:00


  Preparing metadata (setup.py) ... done


  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [2]:
#!/usr/bin/env python3
"""
Launcher script for full precision LoRA fine-tuning on H100 GPU (Kaggle).
Optimized for single H100 GPU with full precision training.
"""

import os
import sys

print("="*80)
print("H100 FULL PRECISION LORA MODE")
print("Training with full precision (no quantization)")
print("="*80)
print()

# Now we can import everything else
import json
import torch
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    TrainerCallback,
)

import transformers
# Enable Transformers verbose logging
transformers.logging.set_verbosity_info()

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)
import numpy as np

# ============================================================================
# Configuration
# ============================================================================

@dataclass
class Config:
    # Model settings
    model_name: str = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
    
    # LoRA settings - optimized for H100
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ])
    lora_bias: str = "none"
    
    # Data settings
    dataset_file: str = "data.csv"  # CSV file
    val_split_ratio: float = 0.1
    random_seed: int = 42
    max_seq_length: int = 2048
    
    # Training settings - optimized for H100
    output_dir: str = "./outputs"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 4  # Increased for H100
    per_device_eval_batch_size: int = 1   # FIX: Reduced to 1 to save eval memory
    gradient_accumulation_steps: int = 4  # Reduced due to larger batch size
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    
    # Logging and evaluation
    logging_steps: int = 10
    eval_steps: int = 100
    save_steps: int = 100
    save_total_limit: int = 3
    
    # Optimization settings - H100 optimized
    optim: str = "adamw_torch_fused"  # Fused optimizer for H100
    gradient_checkpointing: bool = True
    bf16: bool = True  # H100 supports bf16 natively
    fp16: bool = False
    
    # Resume settings
    resume_from_checkpoint: Optional[str] = None
    
    # DeepSpeed settings (optional)
    deepspeed: Optional[str] = None

# ============================================================================
# Data Processing
# ============================================================================

def format_instruction(sample: Dict) -> str:
    """Format the sample into instruction format."""
    prompt = sample["prompt"]
    response = sample["response"]
    
    formatted = f"""<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""
    
    return formatted

def preprocess_function(examples: Dict, tokenizer: AutoTokenizer, max_length: int) -> Dict:
    """Tokenize and prepare the dataset."""
    formatted_texts = [format_instruction(
        {"prompt": p, "response": r}
    ) for p, r in zip(examples["prompt"], examples["response"])]
    
    tokenized = tokenizer(
        formatted_texts,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors=None,
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

def load_and_prepare_data(config: Config, tokenizer: AutoTokenizer):
    """Load and prepare train and validation datasets from a CSV file."""
    print(f"Loading dataset from {config.dataset_file}...")

    # Load CSV file
    try:
        df = pd.read_csv(config.dataset_file)
        print(f"Successfully loaded {len(df)} examples from CSV file")
    except Exception as e:
        raise ValueError(f"Error loading CSV file: {e}")
    
    print(f"Available columns: {df.columns.tolist()}")

    # Validate and rename columns based on what's available
    # Support multiple common column name patterns
    column_mapping = {}
    
    # Check for prompt/input columns
    if 'problem' in df.columns:
        column_mapping['problem'] = 'prompt'
    elif 'question' in df.columns:
        column_mapping['question'] = 'prompt'
    elif 'input' in df.columns:
        column_mapping['input'] = 'prompt'
    elif 'prompt' in df.columns:
        column_mapping['prompt'] = 'prompt'
    else:
        raise ValueError(
            f"Dataset must contain one of: 'problem', 'question', 'input', or 'prompt' columns. "
            f"Found: {df.columns.tolist()}"
        )
    
    # Check for response/output columns
    if 'solution' in df.columns:
        column_mapping['solution'] = 'response'
    elif 'answer' in df.columns:
        column_mapping['answer'] = 'response'
    elif 'output' in df.columns:
        column_mapping['output'] = 'response'
    elif 'response' in df.columns:
        column_mapping['response'] = 'response'
    elif 'completion' in df.columns:
        column_mapping['completion'] = 'response'
    else:
        raise ValueError(
            f"Dataset must contain one of: 'solution', 'answer', 'output', 'response' or 'completion' columns. "
            f"Found: {df.columns.tolist()}"
        )

    # Select and rename columns
    original_columns = list(column_mapping.keys())
    df = df[original_columns].rename(columns=column_mapping)
    
    # Ensure data types are strings and handle missing values
    df['prompt'] = df['prompt'].fillna('').astype(str)
    df['response'] = df['response'].fillna('').astype(str)
    
    # Remove empty rows
    initial_len = len(df)
    df = df[(df['prompt'].str.strip() != '') & (df['response'].str.strip() != '')]
    final_len = len(df)
    
    if final_len < initial_len:
        print(f"Removed {initial_len - final_len} empty rows. Final dataset size: {final_len}")

    if len(df) == 0:
        raise ValueError("No valid data found after cleaning. Please check your CSV file.")

    full_dataset = Dataset.from_pandas(df, preserve_index=False)
    print(f"Total dataset size: {len(full_dataset)}")
    
    split_dataset = full_dataset.train_test_split(
        test_size=config.val_split_ratio,
        seed=config.random_seed,
        shuffle=True
    )
    
    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]
    
    print(f"Train dataset size: {len(train_dataset)} ({(1-config.val_split_ratio)*100:.1f}%)")
    print(f"Validation dataset size: {len(val_dataset)} ({config.val_split_ratio*100:.1f}%)")
    
    train_dataset = train_dataset.map(
        lambda x: preprocess_function(x, tokenizer, config.max_seq_length),
        batched=True,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train dataset",
    )
    
    val_dataset = val_dataset.map(
        lambda x: preprocess_function(x, tokenizer, config.max_seq_length),
        batched=True,
        remove_columns=val_dataset.column_names,
        desc="Tokenizing validation dataset",
    )
    
    return train_dataset, val_dataset

# ============================================================================
# Model Setup with LoRA (Full Precision)
# ============================================================================

def setup_model_and_tokenizer(config: Config):
    """Load and prepare model with LoRA for full precision fine-tuning."""
    # Clear any cached memory first
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        trust_remote_code=True,
        padding_side="right",
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    dtype = torch.bfloat16 if config.bf16 else (torch.float16 if config.fp16 else torch.float32)
    
    print("Loading base model in full precision...")
    # Load model without quantization for full precision training
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map="auto",  # Auto device mapping for single GPU
        low_cpu_mem_usage=True,
        attn_implementation="flash_attention_2",  # Use Flash Attention 2 for H100
    )
    
    # FIX: Disable cache for both training and evaluation to prevent OOM
    model.config.use_cache = False
    model.generation_config.use_cache = False  # FIX: Prevent Trainer from re-enabling cache
    
    print("\nConfiguring LoRA...")
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.lora_target_modules,
        lora_dropout=config.lora_dropout,
        bias=config.lora_bias,
        task_type=TaskType.CAUSAL_LM,
    )
    
    print("Applying LoRA adapters...")
    model = get_peft_model(model, lora_config)
    
    # CRITICAL FIX: Enable gradient checkpointing AFTER applying LoRA
    if config.gradient_checkpointing:
        print("Enabling gradient checkpointing...")
        model.gradient_checkpointing_enable()
        
        # CRITICAL FIX: Enable input to require gradients for gradient checkpointing
        if hasattr(model, 'enable_input_require_grads'):
            model.enable_input_require_grads()
            print("Enabled input_require_grads")
        else:
            # Fallback for older versions
            def make_inputs_require_grad(module, input, output):
                output.requires_grad_(True)
            model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
            print("Enabled input_require_grads (fallback method)")
    
    # Verify trainable parameters
    model.print_trainable_parameters()
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # VERIFICATION: Check that LoRA parameters have requires_grad=True
    lora_params_count = 0
    for name, param in model.named_parameters():
        if 'lora_' in name and param.requires_grad:
            lora_params_count += 1
    
    print(f"\n{'='*80}")
    print(f"Model: {config.model_name}")
    print(f"Precision: Full precision ({dtype})")
    print(f"Attention: Flash Attention 2")
    print(f"LoRA Configuration:")
    print(f"  - Rank (r): {config.lora_r}")
    print(f"  - Alpha: {config.lora_alpha}")
    print(f"  - Dropout: {config.lora_dropout}")
    print(f"  - Target modules: {', '.join(config.lora_target_modules)}")
    print(f"\nParameter Statistics:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")
    print(f"LoRA parameters with grad enabled: {lora_params_count}")
    print(f"Gradient checkpointing: {'Enabled' if config.gradient_checkpointing else 'Disabled'}")
    print(f"{'='*80}\n")
    
    # Final verification
    if trainable_params == 0:
        raise RuntimeError("No trainable parameters found! Check LoRA configuration.")
    
    print("Model ready for H100 training\n")
    
    return model, tokenizer

# ============================================================================
# Custom Callbacks
# ============================================================================

class MetricsCallback(TrainerCallback):
    """Callback to log metrics clearly."""
    
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.steps = []
        self.eval_steps = []
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
                
                print(f"\n{'='*80}")
                print(f"Step: {state.global_step}/{state.max_steps}")
                print(f"Epoch: {state.epoch:.2f}/{args.num_train_epochs}")
                print(f"Training Loss: {logs['loss']:.4f}")
                
                if "learning_rate" in logs:
                    print(f"Learning Rate: {logs['learning_rate']:.2e}")
                    
            if "eval_loss" in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps.append(state.global_step)
                
                print(f"\n{'='*80}")
                print(f"EVALUATION - Step: {state.global_step}")
                print(f"Validation Loss: {logs['eval_loss']:.4f}")
                
                if len(self.eval_losses) > 1:
                    prev_loss = self.eval_losses[-2]
                    improvement = prev_loss - logs['eval_loss']
                    print(f"Change from previous: {improvement:+.4f}")
                    
                if "eval_runtime" in logs:
                    print(f"Eval Runtime: {logs['eval_runtime']:.2f}s")
                if "eval_samples_per_second" in logs:
                    print(f"Eval Samples/sec: {logs['eval_samples_per_second']:.2f}")
                    
                print(f"{'='*80}\n")
    
    def on_train_end(self, args, state, control, **kwargs):
        """Print summary statistics at the end of training."""
        if self.train_losses:
            print(f"\n{'='*80}")
            print("TRAINING SUMMARY")
            print(f"{'='*80}")
            print(f"Initial Training Loss: {self.train_losses[0]:.4f}")
            print(f"Final Training Loss: {self.train_losses[-1]:.4f}")
            print(f"Training Loss Reduction: {self.train_losses[0] - self.train_losses[-1]:.4f}")
            
        if self.eval_losses:
            print(f"\nInitial Validation Loss: {self.eval_losses[0]:.4f}")
            print(f"Final Validation Loss: {self.eval_losses[-1]:.4f}")
            print(f"Validation Loss Reduction: {self.eval_losses[0] - self.eval_losses[-1]:.4f}")
            print(f"Best Validation Loss: {min(self.eval_losses):.4f}")
            print(f"{'='*80}\n")


# FIX: Custom callback to track best model by saving its path, instead of
# loading it into memory (which causes OOM when load_best_model_at_end=True).
class BestModelCallback(TrainerCallback):
    """Tracks the best checkpoint path by eval_loss without loading it into memory."""

    def __init__(self):
        self.best_eval_loss = float("inf")
        self.best_checkpoint_path = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        eval_loss = metrics.get("eval_loss")
        if eval_loss is None:
            return
        if eval_loss < self.best_eval_loss:
            self.best_eval_loss = eval_loss
            # The latest checkpoint saved at this step
            self.best_checkpoint_path = os.path.join(
                args.output_dir, f"checkpoint-{state.global_step}"
            )
            print(f"\n[BestModelCallback] New best eval_loss: {self.best_eval_loss:.4f} "
                  f"at step {state.global_step}")
            print(f"[BestModelCallback] Best checkpoint: {self.best_checkpoint_path}\n")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n{'='*80}")
        print(f"BEST MODEL SUMMARY")
        print(f"Best eval_loss: {self.best_eval_loss:.4f}")
        print(f"Best checkpoint path: {self.best_checkpoint_path}")
        print(f"{'='*80}\n")

# ============================================================================
# Training
# ============================================================================

def train(config: Config):
    """Main training function."""
    
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\n{'='*80}")
        print(f"H100 GPU CONFIGURATION")
        print(f"{'='*80}")
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} - {props.total_memory / 1024**3:.2f} GB")
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"{'='*80}\n")
    
    print("\nLoading model and tokenizer with full precision LoRA...")
    model, tokenizer = setup_model_and_tokenizer(config)
    
    print("Loading and preparing datasets...")
    train_dataset, val_dataset = load_and_prepare_data(config, tokenizer)
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        return_tensors="pt",
    )
    
    training_args = TrainingArguments(
        output_dir=config.output_dir,
        num_train_epochs=config.num_train_epochs,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_eval_batch_size,  # FIX: now 1
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_ratio=config.warmup_ratio,
        lr_scheduler_type="cosine",
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        logging_steps=config.logging_steps,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=False,       # FIX: Prevents double model load causing OOM
        metric_for_best_model="eval_loss",  # Still tracked by BestModelCallback
        greater_is_better=False,
        bf16=config.bf16,
        fp16=config.fp16,
        optim=config.optim,
        report_to="none",
        remove_unused_columns=False,
        deepspeed=config.deepspeed,
        dataloader_pin_memory=True,
        dataloader_num_workers=4,
        dataloader_persistent_workers=False,  # FIX: Prevents worker memory accumulation
        eval_accumulation_steps=4,            # FIX: Accumulate eval in small chunks
        # H100 specific optimizations
        tf32=True,
    )
    
    metrics_callback = MetricsCallback()
    best_model_callback = BestModelCallback()  # FIX: Replaces load_best_model_at_end
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[metrics_callback, best_model_callback],
    )
    
    checkpoint = None
    if config.resume_from_checkpoint is not None:
        checkpoint = config.resume_from_checkpoint
        print(f"\n{'='*80}")
        print(f"RESUMING FROM CHECKPOINT: {checkpoint}")
        print(f"{'='*80}\n")
    
    effective_batch_size = (config.per_device_train_batch_size * 
                           config.gradient_accumulation_steps)
    
    print(f"\n{'='*80}")
    print("TRAINING CONFIGURATION")
    print(f"{'='*80}")
    print(f"Training Mode: Full Precision LoRA")
    print(f"GPU: H100 (Single GPU)")
    print(f"Precision: {torch.bfloat16 if config.bf16 else 'FP32'}")
    print(f"Flash Attention: Enabled")
    print(f"Dataset file: {config.dataset_file}")
    print(f"Train/Val split: {(1-config.val_split_ratio)*100:.0f}% / {config.val_split_ratio*100:.0f}%")
    print(f"Random seed: {config.random_seed}")
    print(f"Total training samples: {len(train_dataset)}")
    print(f"Total validation samples: {len(val_dataset)}")
    print(f"Batch size per device (train): {config.per_device_train_batch_size}")
    print(f"Batch size per device (eval): {config.per_device_eval_batch_size}  [reduced to prevent OOM]")
    print(f"Eval accumulation steps: 4  [chunked to prevent OOM]")
    print(f"Gradient accumulation steps: {config.gradient_accumulation_steps}")
    print(f"Effective batch size: {effective_batch_size}")
    print(f"Number of epochs: {config.num_train_epochs}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Optimizer: {config.optim}")
    print(f"Warmup ratio: {config.warmup_ratio}")
    print(f"Max sequence length: {config.max_seq_length}")
    print(f"Gradient checkpointing: {config.gradient_checkpointing}")
    print(f"load_best_model_at_end: False  [disabled to prevent OOM, BestModelCallback used instead]")
    print(f"dataloader_persistent_workers: False  [disabled to prevent memory accumulation]")
    
    total_steps = (len(train_dataset) // effective_batch_size) * config.num_train_epochs
    print(f"Estimated total steps: {total_steps}")
    print(f"Eval every {config.eval_steps} steps")
    print(f"Save every {config.save_steps} steps")
    print(f"{'='*80}\n")
    
    print("Starting training...\n")
    trainer.train(resume_from_checkpoint=checkpoint)
    
    print("\nSaving final LoRA adapters...")
    final_model_path = os.path.join(config.output_dir, "final_model")
    model.save_pretrained(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    print(f"Final LoRA adapters saved to: {final_model_path}")
    print("Note: To use the model, load the base model and then load these adapters.")
    
    print("\n✓ Training completed successfully!")
    
    print("\nRunning final evaluation...")
    eval_results = trainer.evaluate()
    print(f"\n{'='*80}")
    print("FINAL EVALUATION RESULTS")
    print(f"{'='*80}")
    for key, value in eval_results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    print(f"{'='*80}\n")

# ============================================================================
# Main
# ============================================================================

def main():
    config = Config(
        # CSV file path
        dataset_file="/kaggle/input/datasets/jeannkouagou/aimo3-tool-integrated-reasoning/data.csv",
        val_split_ratio=0.025,
        random_seed=42,
        
        # Enhanced LoRA for full precision H100 training
        lora_r=16,
        lora_alpha=32,
        lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        
        # Training settings - optimized for H100
        output_dir="./qwen_lora_outputs",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,    # FIX: Reduced to 1 for eval memory safety
        gradient_accumulation_steps=8,   # Effective batch = 2*8 = 16
        learning_rate=2e-4,
        max_seq_length=8192,             # Full length for H100
        
        # Evaluation
        eval_steps=250,
        save_steps=250,
        logging_steps=10,
        
        # Optimization - H100 optimized
        gradient_checkpointing=True,
        bf16=True,                       # Native bf16 support on H100
        optim="adamw_torch_fused",       # Fused optimizer for H100
        
        resume_from_checkpoint='/kaggle/input/datasets/secondsyndicate/qwen-finetuning-checkpoints/qwen_lora_outputs/checkpoint-6750',
    )
    
    train(config)

if __name__ == "__main__":
    main()

H100 FULL PRECISION LORA MODE
Training with full precision (no quantization)



loading configuration file /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1/config.json


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at


H100 GPU CONFIGURATION
GPU: NVIDIA H100 80GB HBM3 - 79.18 GB
Compute Capability: 9.0


Loading model and tokenizer with full precision LoRA...


loading configuration file /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1/config.json


...

loading configuration file /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1/config.json


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`


chat template saved in ./qwen_lora_outputs/checkpoint-8750/chat_template.jinja


tokenizer config file saved in ./qwen_lora_outputs/checkpoint-8750/tokenizer_config.json
